# Ejercicio 4.2 · Self-RAG simplificado sobre Fórmula 1

Tres jueces, todos `llama3.2`, sin fine-tune. El primero decide si hace falta buscar (`retrieve` / `no_retrieve`). El segundo, IsRel, deja pasar solo los fragmentos que de verdad responden. El tercero, IsSup, lee la respuesta ya escrita y dice si los fragmentos la sostienen. Si el grado es `not_supported`, el grafo regenera nombrando las afirmaciones que sobran y, si sigue sin soporte, busca en la web.

La búsqueda es la de los ejercicios 3.2 y 3.3: `qwen3-embedding:0.6b`, BM25, fusión RRF con pesos 0.5/0.5 y `bge-reranker-v2-m3` en local. No hay API de pago. La web del fallback es DuckDuckGo.

El corpus (`data/corpus.jsonl`) son 26 apuntes cerrados al final de 2024. Hay hechos que este modelo no vivió, como el título de constructores de McLaren o el debut de Colapinto, y huecos puestos a propósito: clientes de Ferrari, la cifra del cost cap, las victorias de Alonso.


## 1. Índice

Ollama tiene que estar en marcha, con `llama3.2` y `qwen3-embedding:0.6b`. La primera pasada embebe el corpus y carga el reranker. `reconstruir=True` repite los vectores.


In [1]:
import sys
from pathlib import Path

raiz = Path.cwd()
if not (raiz / "selfrag.py").exists():
    raise SystemExit("Ejecuta el notebook desde la carpeta self-rag")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

import selfrag

selfrag.autocomprobar()
selfrag.cargar_entorno()
resumen = selfrag.preparar_indice(reconstruir=False)
estado = "nuevo" if resumen["reconstruido"] else "reutilizado"
print(f"{resumen['documentos']} documentos, {resumen['fragmentos']} fragmentos. Índice {estado}.")
print(f"Web de respaldo: {'DuckDuckGo' if selfrag.web_disponible() else 'no'}")


BGE cargado en 1.1 s
26 documentos, 26 fragmentos. Índice reutilizado.
Web de respaldo: DuckDuckGo


## 2. Retrieval adaptativo

Diez preguntas. Seis piden un hecho del corpus: tres de 2024, que `llama3.2` no tiene en el entrenamiento, y tres que sí puede saber (puntos, Senna, halo). Las otras cuatro no necesitan documentos: un saludo, una multiplicación, una traducción y una despedida.

El prompt pide `retrieve` para cualquier hecho, aunque el modelo crea saberlo. Luego se responde dos veces: con el top 4 del híbrido reordenado y sin documentos. Una búsqueda evitada cuenta como ahorro solo si la respuesta sin documentos sigue siendo válida.


In [2]:
import time

ADAPTATIVAS = [
    {
        "id": "titulos-2024",
        "esperada": "retrieve",
        "pregunta": "¿Cuántos campeonatos de pilotos sumaba Max Verstappen al cerrar 2024?",
        "anclas": ["cuatro", "cuarto", "4 titulo", "4 campeonato"],
    },
    {
        "id": "constructores",
        "esperada": "retrieve",
        "pregunta": "¿Qué equipo ganó el campeonato de constructores en 2024?",
        "anclas": ["mclaren"],
    },
    {
        "id": "colapinto",
        "esperada": "retrieve",
        "pregunta": "¿Quién sustituyó a Logan Sargeant en Williams durante 2024?",
        "anclas": ["colapinto"],
    },
    {
        "id": "puntos",
        "esperada": "retrieve",
        "pregunta": "¿Cuántos puntos recibe el ganador de un gran premio?",
        "anclas": ["25", "veinticinco"],
    },
    {
        "id": "senna",
        "esperada": "retrieve",
        "pregunta": "¿En qué circuito murió Ayrton Senna?",
        "anclas": ["imola", "san marino"],
    },
    {
        "id": "halo",
        "esperada": "retrieve",
        "pregunta": "¿Desde qué año es obligatorio el halo?",
        "anclas": ["2018"],
    },
    {"id": "hola", "esperada": "no_retrieve", "pregunta": "Hola, ¿qué tal?", "anclas": []},
    {"id": "producto", "esperada": "no_retrieve", "pregunta": "¿Cuánto es 12 por 8?", "anclas": ["96"]},
    {
        "id": "traduccion",
        "esperada": "no_retrieve",
        "pregunta": "Traduce al inglés: buenos días.",
        "anclas": ["good morning", "good day"],
    },
    {"id": "adios", "esperada": "no_retrieve", "pregunta": "Gracias, hemos terminado.", "anclas": []},
]

RUIDO = ("verstappen", "mclaren", "senna", "maylander", "imola", "colapinto", "25 puntos")


def recorte(texto: str, n: int = 200) -> str:
    limpio = " ".join(texto.split())
    if len(limpio) <= n:
        return limpio
    return limpio[: n - 1] + "…"


def calidad_adaptativa(caso: dict, respuesta: str) -> bool:
    if caso["anclas"]:
        return selfrag.contiene_ancla(respuesta, caso["anclas"])
    if not respuesta.strip() or selfrag.parece_abstencion(respuesta):
        return False
    base = selfrag.normalizar(respuesta)
    return not any(ruido in base for ruido in RUIDO)


adaptativas = []
for caso in ADAPTATIVAS:
    selfrag.llamadas.reset()
    inicio = time.perf_counter()
    decision = selfrag.clasificar_retrieval(caso["pregunta"])
    busqueda = selfrag.hibrido_rerank(caso["pregunta"], k_pool=8, k_final=4)
    con = selfrag.generar_respuesta(caso["pregunta"], busqueda["docs"])
    sin = selfrag.generar_respuesta(caso["pregunta"], [], charla=True)
    fila = {
        "id": caso["id"],
        "esperada": caso["esperada"],
        "decision": decision,
        "con": con,
        "sin": sin,
        "ok_con": calidad_adaptativa(caso, con),
        "ok_sin": calidad_adaptativa(caso, sin),
        "llamadas": selfrag.llamadas.n,
        "ms": (time.perf_counter() - inicio) * 1000,
        "pregunta": caso["pregunta"],
    }
    adaptativas.append(fila)
    marca_con = "sí" if fila["ok_con"] else "no"
    marca_sin = "sí" if fila["ok_sin"] else "no"
    print(
        f"{fila['id']:<16} esperada {fila['esperada']:<12} modelo {fila['decision']:<12} "
        f"con {marca_con:<3} sin {marca_sin:<3} {fila['ms']:.0f} ms"
    )

aciertos = sum(fila["decision"] == fila["esperada"] for fila in adaptativas)
evitadas = [fila for fila in adaptativas if fila["decision"] == "no_retrieve"]
mantenidas = [fila for fila in evitadas if fila["ok_sin"]]
perdidas = [
    fila
    for fila in adaptativas
    if fila["esperada"] == "retrieve" and fila["decision"] == "no_retrieve" and not fila["ok_sin"]
]
print()
print(f"Clasificador: {aciertos}/{len(adaptativas)} coinciden con la etiqueta.")
print(f"Retrieval evitado: {len(evitadas)}/{len(adaptativas)}.")
print(f"De esas, la respuesta sin documentos sigue valiendo: {len(mantenidas)}/{len(evitadas) or 0}.")
print(f"Hechos del corpus saltados y perdidos: {len(perdidas)}.")
print()
for fila in adaptativas:
    print(fila["id"], "·", fila["pregunta"])
    print("  con:", recorte(fila["con"]))
    print("  sin:", recorte(fila["sin"]))


titulos-2024     esperada retrieve     modelo retrieve     con no  sin sí  3817 ms
constructores    esperada retrieve     modelo retrieve     con sí  sin no  5042 ms
colapinto        esperada retrieve     modelo retrieve     con sí  sin no  5050 ms
puntos           esperada retrieve     modelo retrieve     con sí  sin no  4588 ms
senna            esperada retrieve     modelo retrieve     con sí  sin sí  5240 ms
halo             esperada retrieve     modelo retrieve     con sí  sin no  5058 ms
hola             esperada no_retrieve  modelo no_retrieve  con no  sin sí  4552 ms
producto         esperada no_retrieve  modelo no_retrieve  con no  sin sí  4761 ms
traduccion       esperada no_retrieve  modelo no_retrieve  con sí  sin sí  4358 ms
adios            esperada no_retrieve  modelo no_retrieve  con sí  sin sí  4362 ms

Clasificador: 10/10 coinciden con la etiqueta.
Retrieval evitado: 4/10.
De esas, la respuesta sin documentos sigue valiendo: 4/4.
Hechos del corpus saltados y perdidos: 

## 3. IsRel frente al híbrido con reranker

Se piden 8 candidatos, BGE los reordena y se quedan 5, igual que en 3.3. IsRel etiqueta esos 5. El reranker siempre rellena el cupo. IsRel puede dejar la lista vacía.

En las dos preguntas de dentro se mira si el fragmento con la ancla sobrevive. En la de los clientes y en la de MotoGP no hay ancla: un buen filtro las vacía, y el generador tiene que abstenerse en vez de apoyarse en un fragmento de Ferrari o de otro gran premio que no responde.


In [3]:
import re

COMPARAR = [
    {
        "id": "constructores",
        "tipo": "en_corpus",
        "pregunta": "¿Qué escudería se llevó el mundial de equipos en 2024?",
        "anclas": ["mclaren"],
    },
    {
        "id": "colapinto",
        "tipo": "en_corpus",
        "pregunta": "¿Qué piloto argentino debutó con Williams en Monza en 2024?",
        "anclas": ["colapinto"],
    },
    {
        "id": "clientes",
        "tipo": "alucinacion",
        "pregunta": "¿Cuántos clientes tiene Ferrari?",
        "anclas": [],
    },
    {
        "id": "motogp",
        "tipo": "fuera",
        "pregunta": "¿Quién ganó el mundial de MotoGP en 2023?",
        "anclas": [],
    },
]


def cifra_sospechosa(respuesta: str, pregunta: str) -> bool:
    nuevas = set(re.findall(r"\d{2,}", respuesta)) - set(re.findall(r"\d{2,}", pregunta))
    for cifra in nuevas:
        if len(cifra) == 4 and cifra.startswith(("19", "20")):
            continue
        return True
    return False


def calidad_cerrada(caso: dict, respuesta: str) -> bool:
    if caso["tipo"] == "en_corpus":
        return selfrag.contiene_ancla(respuesta, caso["anclas"])
    return selfrag.parece_abstencion(respuesta) and not cifra_sospechosa(respuesta, caso["pregunta"])


def ancla_en(docs, anclas: list[str]):
    if not anclas:
        return None
    return any(selfrag.contiene_ancla(doc.page_content, anclas) for doc in docs)


comparacion = []
for caso in COMPARAR:
    selfrag.llamadas.reset()
    resultado = selfrag.hibrido_rerank(caso["pregunta"], k_pool=8, k_final=5)
    docs = resultado["docs"]
    inicio = time.perf_counter()
    buenos, detalle = selfrag.filtrar_relevantes(caso["pregunta"], docs)
    t_isrel = time.perf_counter() - inicio
    resp_rerank = selfrag.generar_respuesta(caso["pregunta"], docs)
    resp_isrel = selfrag.generar_respuesta(caso["pregunta"], buenos)
    acuerdo = sum((fila["nota"] >= 0.5) == fila["relevant"] for fila in detalle)
    fila = {
        "id": caso["id"],
        "tipo": caso["tipo"],
        "pregunta": caso["pregunta"],
        "detalle": detalle,
        "descartados": sum(not item["relevant"] for item in detalle),
        "n": len(detalle),
        "acuerdo": acuerdo,
        "ancla_rerank": ancla_en(docs, caso["anclas"]),
        "ancla_isrel": ancla_en(buenos, caso["anclas"]),
        "ok_rerank": calidad_cerrada(caso, resp_rerank),
        "ok_isrel": calidad_cerrada(caso, resp_isrel),
        "resp_rerank": resp_rerank,
        "resp_isrel": resp_isrel,
        "t_busqueda": resultado["t_busqueda"],
        "t_rerank": resultado["t_rerank"],
        "t_isrel": t_isrel,
        "notas_rel": [item["nota"] for item in detalle if item["relevant"]],
        "notas_irr": [item["nota"] for item in detalle if not item["relevant"]],
    }
    comparacion.append(fila)
    print(
        f"{fila['id']:<16} descarta {fila['descartados']}/{fila['n']}  "
        f"acuerdo con nota≥0.5 {fila['acuerdo']}/{fila['n']}  "
        f"rerank {resultado['t_rerank']*1000:.0f} ms  IsRel {t_isrel*1000:.0f} ms"
    )
    if caso["anclas"]:
        print(f"  ancla en el top rerank: {'sí' if fila['ancla_rerank'] else 'no'}   tras IsRel: {'sí' if fila['ancla_isrel'] else 'no'}")
    print(f"  respuesta rerank {'sí' if fila['ok_rerank'] else 'no'}   IsRel {'sí' if fila['ok_isrel'] else 'no'}")
    for item in detalle:
        marca = "sí" if item["relevant"] else "no"
        print(f"    {marca}  {item['nota']:.2f}  {item['titulo']}")
    print("  rerank:", recorte(resp_rerank))
    print("  IsRel: ", recorte(resp_isrel))
    print()


constructores    descarta 5/5  acuerdo con nota≥0.5 2/5  rerank 569 ms  IsRel 2463 ms
  ancla en el top rerank: sí   tras IsRel: no
  respuesta rerank no   IsRel no
    no  0.96  Ferrari
    no  0.91  Equipos de 2024
    no  0.59  Pilotos 2024
    no  0.39  Constructores 2024
    no  0.04  El cambio en Williams
  rerank: No tengo información específica sobre el ganador del mundial de equipos en 2024.
  IsRel:  No tengo información sobre el resultado del mundial de equipos en 2024, ya que no tengo acceso a datos futuros o información que no esté actualizada.

colapinto        descarta 2/5  acuerdo con nota≥0.5 4/5  rerank 566 ms  IsRel 2487 ms
  ancla en el top rerank: sí   tras IsRel: sí
  respuesta rerank sí   IsRel sí
    sí  1.00  El cambio en Williams
    sí  0.78  Equipos de 2024
    no  0.12  Fernando Alonso
    no  0.08  Ferrari
    sí  0.03  Pilotos 2024
  rerank: El piloto argentino que debutó con Williams en Monza en 2024 fue Franco Colapinto.
  IsRel:  El piloto argentino qu

## 4. El grafo

`construir_grafo` encadena las tres decisiones. `confidence` sale del último IsSup: `fully_supported` es high, `partially_supported` es medium, `not_supported` es low.

`llama3.2` falla en dos sitios y el grafo los corrige antes de enrutar: si la respuesta dice que el dato no consta y no añade una cifra, el grado pasa a `fully_supported`; si no quedó ningún fragmento y la respuesta aun así afirma un dato, el grado pasa a `not_supported`.

```
clasificar
  retrieve     → recuperar (híbrido + BGE) → IsRel → generar
  no_retrieve  → generar sin documentos
generar → IsSup
  not_supported, con fragmentos, aún sin regenerar → regenerar → IsSup
  not_supported y la web está disponible          → DuckDuckGo → IsSup
  en otro caso                                    → cerrar
```


In [4]:
print(selfrag.mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	clasificar(clasificar)
	recuperar(recuperar)
	filtrar(filtrar)
	generar(generar)
	verificar(verificar)
	regenerar(regenerar)
	web(web)
	cerrar(cerrar)
	__end__([<p>__end__</p>]):::last
	__start__ --> clasificar;
	clasificar -.-> generar;
	clasificar -.-> recuperar;
	filtrar --> generar;
	generar --> verificar;
	recuperar --> filtrar;
	regenerar --> verificar;
	verificar -.-> cerrar;
	verificar -.-> regenerar;
	verificar -.-> web;
	web -.-> cerrar;
	web -.-> verificar;
	cerrar --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 5. Evaluación

Nueve preguntas. Tres tienen la respuesta escrita en un apunte. Tres piden una cifra que el corpus no da y que el modelo tiende a rellenar. Tres se salen del tema.

La tasa de detección es el primer IsSup, antes de regenerar o de salir a la web. La nota de 0 a 2 es un proxy: 2 si trae el dato del corpus o si se abstiene en una trampa, 1 si la web sostiene una respuesta de fuera, 0 si afirma un dato que el corpus no trae.


In [5]:
EVALUAR = [
    {
        "id": "constructores",
        "tipo": "en_corpus",
        "pregunta": "¿Quién ganó el campeonato de constructores de 2024?",
        "anclas": ["mclaren"],
    },
    {
        "id": "colapinto",
        "tipo": "en_corpus",
        "pregunta": "¿Qué argentino ocupó el asiento de Logan Sargeant a partir de Monza 2024?",
        "anclas": ["colapinto"],
    },
    {
        "id": "monaco",
        "tipo": "en_corpus",
        "pregunta": "¿Cuántas vueltas tiene el Gran Premio de Mónaco?",
        "anclas": ["78"],
    },
    {
        "id": "clientes",
        "tipo": "alucinacion",
        "pregunta": "¿Cuántos clientes tiene Ferrari?",
        "anclas": [],
    },
    {
        "id": "costcap",
        "tipo": "alucinacion",
        "pregunta": "¿De cuántos dólares fue el límite de costes en 2024?",
        "anclas": [],
    },
    {
        "id": "alonso",
        "tipo": "alucinacion",
        "pregunta": "¿Cuántas victorias en grandes premios tiene Fernando Alonso?",
        "anclas": [],
    },
    {
        "id": "motogp",
        "tipo": "fuera",
        "pregunta": "¿Quién ganó el mundial de MotoGP en 2023?",
        "anclas": [],
    },
    {
        "id": "mongolia",
        "tipo": "fuera",
        "pregunta": "¿Cuál es la capital de Mongolia?",
        "anclas": [],
    },
    {
        "id": "walrus",
        "tipo": "fuera",
        "pregunta": "¿En qué versión de Python apareció el operador walrus?",
        "anclas": [],
    },
]


def nota_final(caso: dict, resultado: dict) -> tuple[int, str]:
    respuesta = resultado["respuesta"]
    if caso["tipo"] == "en_corpus":
        if selfrag.contiene_ancla(respuesta, caso["anclas"]):
            return 2, "trae el dato del corpus"
        return 0, "no trae el dato del corpus"
    if selfrag.parece_abstencion(respuesta) and not cifra_sospechosa(respuesta, caso["pregunta"]):
        return 2, "se abstiene"
    if resultado["fuente"] == "web" and resultado["soporte"] == "fully_supported":
        return 1, "la web la sostiene"
    if resultado["fuente"] == "web":
        return 1, "sale de la web"
    return 0, "afirma un dato que el corpus no trae"


evaluacion = []
for caso in EVALUAR:
    print(caso["id"], "·", caso["pregunta"], flush=True)
    resultado = selfrag.responder(caso["pregunta"])
    nota, motivo = nota_final(caso, resultado)
    fila = {**caso, **resultado, "nota": nota, "motivo": motivo}
    evaluacion.append(fila)
    print(
        f"  {resultado['decision']:<12} inicial {resultado['soporte_inicial']:<22} "
        f"final {resultado['soporte']:<22} {resultado['confidence']:<6} "
        f"fuente {resultado['fuente']:<16} {resultado['llamadas_llm']} llamadas  "
        f"{resultado['latencia_s']*1000:.0f} ms  nota {nota} ({motivo})"
    )
    print("  traza:", " → ".join(resultado["traza"]))
    print("  ", recorte(resultado["respuesta"], 280))
    print()


constructores · ¿Quién ganó el campeonato de constructores de 2024?
  retrieve     inicial partially_supported    final partially_supported    medium fuente corpus           8 llamadas  6419 ms  nota 2 (trae el dato del corpus)
  traza: retrieval:retrieve → hibrido+rerank:5 → isrel:1/5 → generar → issup:partially_supported
   McLaren ganó el campeonato de constructores de 2024.

colapinto · ¿Qué argentino ocupó el asiento de Logan Sargeant a partir de Monza 2024?
  retrieve     inicial partially_supported    final partially_supported    medium fuente corpus           8 llamadas  6615 ms  nota 2 (trae el dato del corpus)
  traza: retrieval:retrieve → hibrido+rerank:5 → isrel:2/5 → generar → issup:partially_supported
   El piloto argentino que ocupó el asiento de Logan Sargeant a partir de Monza 2024 fue Franco Colapinto.

monaco · ¿Cuántas vueltas tiene el Gran Premio de Mónaco?
  retrieve     inicial partially_supported    final partially_supported    medium fuente corpus           8 l

## 6. Sonda del detector

El generador del grafo tiene prohibido inventar, así que a menudo no hay alucinación que marcar. Esta sonda pide la cifra con un prompt laxo, el que sí se sale del corpus, y pasa esa respuesta por IsSup junto a los fragmentos recuperados. Aquí se mide el juez, no la cadena estricta.


In [6]:
sonda = []
for caso in EVALUAR:
    if caso["tipo"] != "alucinacion":
        continue
    docs = selfrag.hibrido_rerank(caso["pregunta"], k_pool=8, k_final=4)["docs"]
    libre = selfrag.generar_libre(caso["pregunta"], docs)
    juicio = selfrag.verificar_soporte(caso["pregunta"], docs, libre)
    fila = {
        "id": caso["id"],
        "pregunta": caso["pregunta"],
        "libre": libre,
        "soporte": juicio["soporte"],
        "afirmaciones": juicio["afirmaciones"],
        "cifra": cifra_sospechosa(libre, caso["pregunta"]),
    }
    sonda.append(fila)
    print(caso["id"], juicio["soporte"], "cifra nueva" if fila["cifra"] else "sin cifra nueva")
    print(" ", recorte(libre, 280))
    if juicio["afirmaciones"]:
        print("  afirmaciones:", " || ".join(juicio["afirmaciones"]))
    print()


clientes not_supported sin cifra nueva
  La respuesta a la pregunta es que no se menciona el número de clientes de Ferrari en la información proporcionada. Solo se menciona que Ferrari es la única escudería presente en todos los campeonatos del mundo desde 1950 y que su sede está en Maranello, Italia.

costcap not_supported sin cifra nueva
  Lo siento, pero no tengo información sobre el límite de costes en dólares para la temporada 2024 de Fórmula 1. La información proporcionada solo menciona que el límite de costes se aplicó desde la temporada 2021 y que incumplirlo se sanciona con multas y recortes de tiempo en el…

alonso not_supported sin cifra nueva
  No tengo información sobre las victorias en grandes premios de Fernando Alonso.



## 7. Lectura

Los números salen de las celdas de arriba. La nota de calidad es el proxy automático, no un juicio ciego: en una respuesta de dentro cuenta que aparezca la ancla, y en una trampa cuenta que la respuesta se abstenga.


In [7]:
def media(valores: list[float]) -> float:
    return sum(valores) / len(valores) if valores else 0.0


aciertos = sum(fila["decision"] == fila["esperada"] for fila in adaptativas)
evitadas = [fila for fila in adaptativas if fila["decision"] == "no_retrieve"]
mantenidas = [fila for fila in evitadas if fila["ok_sin"]]
perdidas = [
    fila
    for fila in adaptativas
    if fila["esperada"] == "retrieve" and fila["decision"] == "no_retrieve" and not fila["ok_sin"]
]
sabidas = [fila for fila in adaptativas if fila["esperada"] == "retrieve" and fila["ok_sin"]]
empeora = [fila for fila in adaptativas if fila["esperada"] == "no_retrieve" and fila["ok_sin"] and not fila["ok_con"]]

print("Retrieval adaptativo")
print(
    f"El clasificador acierta {aciertos} de {len(adaptativas)}. "
    f"Evita la búsqueda en {len(evitadas)} de {len(adaptativas)}, "
    f"y en {len(mantenidas)} de esas {len(evitadas) or 0} la respuesta sin documentos sigue siendo válida."
)
if perdidas:
    nombres = ", ".join(fila["id"] for fila in perdidas)
    print(f"Se dejó sin documentos un hecho del corpus y la respuesta perdió el dato: {nombres}.")
else:
    print("Ningún hecho del corpus se quedó sin búsqueda y sin el dato.")
if sabidas:
    print(
        "Aun así, "
        + ", ".join(fila["id"] for fila in sabidas)
        + " ya salían bien sin documentos: el modelo se las sabía. Ahí la búsqueda no mejora la frase, la ancla."
    )
if empeora:
    print(
        "En "
        + ", ".join(fila["id"] for fila in empeora)
        + " meter fragmentos empeora la respuesta. Son las que no había que buscar."
    )

dentro = [fila for fila in comparacion if fila["tipo"] == "en_corpus"]
anclas_rerank = sum(bool(fila["ancla_rerank"]) for fila in dentro)
anclas_isrel = sum(bool(fila["ancla_isrel"]) for fila in dentro)
descartados = sum(fila["descartados"] for fila in comparacion)
juzgados = sum(fila["n"] for fila in comparacion)
acuerdos = sum(fila["acuerdo"] for fila in comparacion)
vacias = [fila["id"] for fila in comparacion if fila["descartados"] == fila["n"] and fila["n"]]
mejor = [fila["id"] for fila in comparacion if fila["ok_isrel"] and not fila["ok_rerank"]]
peor = [fila["id"] for fila in comparacion if fila["ok_rerank"] and not fila["ok_isrel"]]
ms_rerank = media([fila["t_rerank"] for fila in comparacion]) * 1000
ms_isrel = media([fila["t_isrel"] for fila in comparacion]) * 1000

print()
print("IsRel y el reranker")
print(
    f"La ancla de las preguntas de dentro está en el top del reranker en {anclas_rerank} de {len(dentro)} "
    f"y sigue después de IsRel en {anclas_isrel} de {len(dentro)}. "
    f"IsRel descarta {descartados} de {juzgados} fragmentos. "
    f"Coincide con el corte de nota 0.5 en {acuerdos} de {juzgados}."
)
print(f"Reordenar tarda de media {ms_rerank:.0f} ms. Etiquetar los mismos fragmentos con el LLM tarda {ms_isrel:.0f} ms.")
if vacias:
    print("Deja la lista vacía en: " + ", ".join(vacias) + ". El reranker, en esas, había rellenado los cinco huecos.")
if mejor:
    print("La respuesta mejora al filtrar en: " + ", ".join(mejor) + ".")
if peor:
    print("La respuesta empeora al filtrar en: " + ", ".join(peor) + ".")
if not mejor and not peor:
    print("En esta muestra la respuesta final no cambia de válida a inválida por el filtro.")
if acuerdos / juzgados >= 0.8 and not vacias:
    print("Con este acuerdo, IsRel repite lo que ya haría un umbral sobre la nota de BGE.")
elif vacias or acuerdos / juzgados < 0.8:
    print(
        "No es el mismo corte que la nota: o vacía preguntas que el top-k siempre rellena, "
        "o se separa del 0.5. Ahí el juez binario no sobra."
    )

detecciones = [fila for fila in evaluacion if fila["soporte_inicial"] == "not_supported"]
trampas = [fila for fila in evaluacion if fila["tipo"] != "en_corpus"]
det_trampas = [fila for fila in trampas if fila["soporte_inicial"] == "not_supported"]
regens = [fila for fila in evaluacion if fila["regenerada"]]
webs = [fila for fila in evaluacion if fila["web_usada"]]
print()
print("Grafo")
print(
    f"Primer IsSup en not_supported: {len(detecciones)} de {len(evaluacion)}. "
    f"En las que no están resueltas por el corpus: {len(det_trampas)} de {len(trampas)}."
)
print(
    f"Latencia media {media([fila['latencia_s'] for fila in evaluacion])*1000:.0f} ms. "
    f"Llamadas LLM medias {media([fila['llamadas_llm'] for fila in evaluacion]):.1f}."
)
print(
    f"Nota media del proxy: {media([fila['nota'] for fila in evaluacion]):.2f} sobre 2. "
    f"Regenera en {len(regens)} y sale a la web en {len(webs)}."
)
conteo = {"high": 0, "medium": 0, "low": 0}
for fila in evaluacion:
    conteo[fila["confidence"]] = conteo.get(fila["confidence"], 0) + 1
print(
    "confidence: "
    + ", ".join(f"{nivel} {conteo.get(nivel, 0)}" for nivel in ("high", "medium", "low"))
    + "."
)

con_cifra = [fila for fila in sonda if fila["cifra"]]
marcadas = [fila for fila in sonda if fila["soporte"] == "not_supported"]
cazadas = [fila for fila in sonda if fila["cifra"] and fila["soporte"] == "not_supported"]
print()
print("Sonda")
print(
    f"De {len(sonda)} respuestas laxas, {len(con_cifra)} meten una cifra que no es un año de la pregunta "
    f"y {len(marcadas)} quedan not_supported. El detector marca {len(cazadas)} de esas cifras."
)


Retrieval adaptativo
El clasificador acierta 10 de 10. Evita la búsqueda en 4 de 10, y en 4 de esas 4 la respuesta sin documentos sigue siendo válida.
Ningún hecho del corpus se quedó sin búsqueda y sin el dato.
Aun así, titulos-2024, senna ya salían bien sin documentos: el modelo se las sabía. Ahí la búsqueda no mejora la frase, la ancla.
En hola, producto meter fragmentos empeora la respuesta. Son las que no había que buscar.

IsRel y el reranker
La ancla de las preguntas de dentro está en el top del reranker en 2 de 2 y sigue después de IsRel en 1 de 2. IsRel descarta 14 de 20 fragmentos. Coincide con el corte de nota 0.5 en 13 de 20.
Reordenar tarda de media 533 ms. Etiquetar los mismos fragmentos con el LLM tarda 2475 ms.
Deja la lista vacía en: constructores. El reranker, en esas, había rellenado los cinco huecos.
En esta muestra la respuesta final no cambia de válida a inválida por el filtro.
No es el mismo corte que la nota: o vacía preguntas que el top-k siempre rellena, o se 